# Automated Crack Detection in UTM Testing
### Binary Image Classification: Damaged vs. Undamaged Specimens
---
**Pipeline summary:**
1. Mount Google Drive and load dataset
2. Temporal-aware train / val / test split (no data leakage)
3. Exploratory Data Analysis
4. `tf.data` input pipeline with augmentation
5. Baseline custom CNN
6. EfficientNetB0 — Phase 1 (head only) → Phase 2 (fine-tuning)
7. Threshold optimisation (Precision-Recall curve)
8. Full test-set evaluation (confusion matrix, ROC, classification report)
9. Grad-CAM saliency visualisation
10. Save model to Drive

In [ ]:
# ── GPU verification ────────────────────────────────────────────────────────
import tensorflow as tf, sys

print("Python        :", sys.version)
print("TensorFlow    :", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print(f"GPU detected  : {gpus[0].name}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
else:
    print("No GPU detected — go to Runtime > Change runtime type > T4 GPU")

print("\nAll devices   :", tf.config.list_physical_devices())

In [ ]:
# ── Install / import all libraries ──────────────────────────────────────────
# (All packages are pre-installed on Colab; nothing extra needed)
import os, math, random, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve, f1_score
)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")
AUTOTUNE = tf.data.AUTOTUNE

print("All imports OK")

## Step 1 — Mount Google Drive
Upload your `Dataset/` folder to Google Drive at the path:
```
MyDrive/crack-detection/Dataset/Damaged/     ← 359 images
MyDrive/crack-detection/Dataset/Undamaged/   ← 209 images
```
Then run the cell below to mount the drive.

In [ ]:
# ── Mount Google Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"
DATASET_DIR = os.path.join(DRIVE_ROOT, "Dataset")

damaged_dir   = os.path.join(DATASET_DIR, "Damaged")
undamaged_dir = os.path.join(DATASET_DIR, "Undamaged")

# Quick sanity check
n_dmg = len(list(pathlib.Path(damaged_dir).glob("*.jpg")))
n_und = len(list(pathlib.Path(undamaged_dir).glob("*.jpg")))
print(f"Damaged images   : {n_dmg}")
print(f"Undamaged images : {n_und}")
print(f"Total            : {n_dmg + n_und}")

## Step 2 — Configuration
All hyper-parameters are centralised here. Edit this cell before running the rest.

In [ ]:
# ── Global configuration ─────────────────────────────────────────────────────

# Input
IMG_SIZE    = 224          # EfficientNetB0 native size
CHANNELS    = 3
IMG_SHAPE   = (IMG_SIZE, IMG_SIZE, CHANNELS)

# Dataset splits (per class, chronological)
TRAIN_FRAC  = 0.70
VAL_FRAC    = 0.15        # remaining 15 % → test

# Training
BATCH_SIZE  = 32
SEED        = 42

# Phase 1 — head only (frozen backbone)
P1_EPOCHS   = 30
P1_LR       = 1e-3

# Phase 2 — fine-tune top layers of backbone
P2_EPOCHS   = 30
P2_LR       = 1e-5
UNFREEZE_FROM = -40       # unfreeze last 40 layers of EfficientNetB0

# Class names (index 0 = Undamaged, index 1 = Damaged)
CLASS_NAMES = ["Undamaged", "Damaged"]

# Class weights — penalise missing a crack 2× more
CLASS_WEIGHTS = {0: 1.0, 1: 2.0}

# Decision threshold (tuned on val set; see threshold optimisation cell)
THRESHOLD   = 0.40        # will be updated after threshold search

# Paths for saving
SAVE_DIR    = os.path.join(DRIVE_ROOT, "saved_models")
os.makedirs(SAVE_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Config loaded.")
print(f"  IMG_SIZE    : {IMG_SIZE}")
print(f"  BATCH_SIZE  : {BATCH_SIZE}")
print(f"  Class names : {CLASS_NAMES}")
print(f"  Class weights: {CLASS_WEIGHTS}")

## Step 3 — Dataset Loading & Temporal Split

**Why temporal (chronological) split?**
All images are sequential video frames. A random split would put near-identical adjacent frames in both train and test, creating **data leakage** and inflating test accuracy. Instead, we split each class chronologically:
- First 70 % → Train
- Next 15 % → Validation
- Last 15 % → Test

In [ ]:
# ── Temporal (chronological) split ──────────────────────────────────────────

def temporal_split(directory, label, train_frac=TRAIN_FRAC, val_frac=VAL_FRAC):
    """
    Returns three lists of (path, label) tuples: train, val, test.
    Files are sorted chronologically (by filename frame number).
    """
    paths = sorted(pathlib.Path(directory).glob("*.jpg"),
                   key=lambda p: int(p.stem.split("_")[1]))
    n = len(paths)
    n_train = int(n * train_frac)
    n_val   = int(n * val_frac)
    train = [(str(p), label) for p in paths[:n_train]]
    val   = [(str(p), label) for p in paths[n_train : n_train + n_val]]
    test  = [(str(p), label) for p in paths[n_train + n_val :]]
    return train, val, test

# Damaged  → label 1 | Undamaged → label 0
dmg_train,  dmg_val,  dmg_test  = temporal_split(damaged_dir,   label=1)
und_train,  und_val,  und_test  = temporal_split(undamaged_dir, label=0)

train_data = dmg_train + und_train
val_data   = dmg_val   + und_val
test_data  = dmg_test  + und_test

# Shuffle train only (no shuffling in val/test — order doesn't matter there)
random.shuffle(train_data)

# Summary
for split, name in [(train_data,"Train"), (val_data,"Val"), (test_data,"Test")]:
    dmg_count = sum(1 for _, l in split if l == 1)
    und_count = sum(1 for _, l in split if l == 0)
    print(f"{name:5s}: {len(split):4d} total  "
          f"| Damaged={dmg_count}  Undamaged={und_count}")

## Step 4 — Exploratory Data Analysis

In [ ]:
# ── EDA: class distribution ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart — full dataset
all_data = train_data + val_data + test_data
counts   = [sum(1 for _, l in all_data if l == 0),
            sum(1 for _, l in all_data if l == 1)]
bars = axes[0].bar(CLASS_NAMES, counts, color=["steelblue", "tomato"],
                   edgecolor="black", width=0.5)
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(cnt), ha="center", fontsize=12, fontweight="bold")
axes[0].set_title("Class Distribution (full dataset)", fontsize=13)
axes[0].set_ylabel("Number of images")
axes[0].set_ylim(0, max(counts) * 1.15)

# Stacked bar — per split
splits_info = {
    "Train" : train_data,
    "Val"   : val_data,
    "Test"  : test_data,
}
split_names  = list(splits_info.keys())
split_und    = [sum(1 for _, l in v if l == 0) for v in splits_info.values()]
split_dmg    = [sum(1 for _, l in v if l == 1) for v in splits_info.values()]
x = range(len(split_names))
b1 = axes[1].bar(x, split_und, color="steelblue", label="Undamaged", edgecolor="black")
b2 = axes[1].bar(x, split_dmg, bottom=split_und, color="tomato",
                 label="Damaged", edgecolor="black")
axes[1].set_xticks(list(x))
axes[1].set_xticklabels(split_names)
axes[1].set_title("Split Distribution", fontsize=13)
axes[1].set_ylabel("Number of images")
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "class_distribution.png"), dpi=150)
plt.show()

print(f"\nTotal train : {len(train_data)}")
print(f"Total val   : {len(val_data)}")
print(f"Total test  : {len(test_data)}")

In [ ]:
# ── EDA: sample images from each class ───────────────────────────────────────
from PIL import Image as PILImage

def show_samples(data, class_label, n=4, title_prefix=""):
    paths = [p for p, l in data if l == class_label]
    # Pick n evenly spaced samples across the class timeline
    idxs  = np.linspace(0, len(paths)-1, n, dtype=int)
    return [paths[i] for i in idxs]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle("Sample Images — Top: Undamaged  |  Bottom: Damaged",
             fontsize=14, fontweight="bold")

all_with_labels = train_data + val_data + test_data
for row, (label, class_name) in enumerate([(0, "Undamaged"), (1, "Damaged")]):
    samples = show_samples(all_with_labels, label, n=4)
    for col, path in enumerate(samples):
        img = PILImage.open(path)
        # Crop centre strip to highlight specimen area
        w, h = img.size
        left, right = int(w * 0.25), int(w * 0.75)
        img_crop = img.crop((left, 0, right, h))
        axes[row][col].imshow(img_crop)
        axes[row][col].set_title(
            f"{class_name}\n{os.path.basename(path)}", fontsize=9)
        axes[row][col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "sample_images.png"), dpi=100)
plt.show()

## Step 5 — tf.data Input Pipeline

The pipeline:
1. Reads JPEG files on-the-fly (no pre-loading into RAM)
2. Resizes from 3840×2176 → 224×224
3. Applies augmentation **only** during training
4. Prefetches batches to keep GPU fully utilized

In [ ]:
# ── Augmentation layer (applied inside the tf.data pipeline) ──────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.08),          # ±8 degrees
    layers.RandomZoom(0.10),              # ±10 %
    layers.RandomContrast(0.20),          # ±20 %
    layers.RandomBrightness(0.20),        # ±20 %
], name="augmentation")

# ── tf.data helper functions ──────────────────────────────────────────────────

def load_and_preprocess(path, label):
    """Read JPEG → decode → resize → float32 [0, 255] (EfficientNet normalises internally)."""
    raw  = tf.io.read_file(path)
    img  = tf.image.decode_jpeg(raw, channels=CHANNELS)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32)       # keep [0,255] — EfficientNetB0 handles normalisation
    return img, label

def make_dataset(data_list, shuffle=False, augment=False):
    """Build a tf.data.Dataset from a list of (path, label) tuples."""
    paths  = [p for p, _ in data_list]
    labels = [l for _, l in data_list]
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(data_list), seed=SEED)
    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    # Cache decoded+resized images in RAM — eliminates Drive I/O on every epoch.
    # 568 images × 224×224×3 × 4 bytes ≈ 342 MB, well within Colab's ~12 GB RAM.
    ds = ds.cache()
    if augment:
        # Augmentation runs AFTER cache so each epoch still gets fresh random transforms.
        ds = ds.map(lambda img, lbl: (data_augmentation(img, training=True), lbl),
                    num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_data, shuffle=True,  augment=True)
val_ds   = make_dataset(val_data,   shuffle=False, augment=False)
test_ds  = make_dataset(test_data,  shuffle=False, augment=False)

# ── Quick sanity check ────────────────────────────────────────────────────────
print("Datasets built:")
for ds, name in [(train_ds, "train"), (val_ds, "val"), (test_ds, "test")]:
    print(f"  {name:6s}: {len(list(ds))} batches")

In [ ]:
# ── Visualise augmentation effect on one training image ──────────────────────
sample_path, sample_label = train_data[0]
raw  = tf.io.read_file(sample_path)
img  = tf.image.decode_jpeg(raw, channels=3)
img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
img  = tf.cast(img, tf.float32)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle(f"Original vs Augmented   |   Class: {CLASS_NAMES[sample_label]}",
             fontsize=13, fontweight="bold")

axes[0][0].imshow(img.numpy().astype("uint8"))
axes[0][0].set_title("Original")
axes[0][0].axis("off")

for i in range(1, 10):
    aug_img = data_augmentation(img, training=True)
    r, c = divmod(i, 5)
    axes[r][c].imshow(aug_img.numpy().clip(0, 255).astype("uint8"))
    axes[r][c].set_title(f"Aug #{i}")
    axes[r][c].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "augmentation_samples.png"), dpi=100)
plt.show()

## Step 6 — Baseline Custom CNN

A small custom CNN is trained first as a **lower-bound benchmark**.  
It uses no pre-trained weights, so its accuracy tells us directly how much value transfer learning adds.

In [ ]:
# ── Baseline CNN definition ──────────────────────────────────────────────────

def build_custom_cnn(input_shape=IMG_SHAPE):
    inputs = keras.Input(shape=input_shape)

    x = layers.Rescaling(1./255)(inputs)   # normalise to [0, 1]

    # Block 1
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 2
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 3
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)

    # Block 4
    x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Classifier head
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)

    model = Model(inputs, outputs, name="CustomCNN_Baseline")
    return model

cnn_baseline = build_custom_cnn()
cnn_baseline.summary()

In [ ]:
# ── Train baseline CNN ────────────────────────────────────────────────────────
cnn_baseline.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ]
)

# Baseline is a benchmark only — 10 epochs is enough to establish the lower bound.
# The dataset is now cached in RAM so each epoch is fast.
cbs_cnn = [
    EarlyStopping(monitor="val_recall", patience=5,
                  mode="max", restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

hist_cnn = cnn_baseline.fit(
    train_ds,
    epochs=10,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=cbs_cnn,
    verbose=1
)

print("\nBaseline CNN training complete.")

## Step 7 — EfficientNetB0 with Transfer Learning

### Phase 1 — Feature Extraction (frozen backbone)
The EfficientNetB0 backbone (ImageNet weights) is completely frozen.  
Only the custom classification head is trained. This quickly adapts the head without risking catastrophic forgetting.

### Phase 2 — Fine-Tuning (partially unfrozen backbone)
The last ~40 layers of the backbone are unfrozen and trained at a very low learning rate (1e-5).  
This gently adapts deep crack-specific feature detectors.

In [ ]:
# ── Build EfficientNetB0 model ────────────────────────────────────────────────

def build_efficientnet(trainable_backbone=False, unfreeze_from=None):
    """
    trainable_backbone : if False, backbone is fully frozen (Phase 1).
                         if True with unfreeze_from, only top layers unfrozen (Phase 2).
    """
    inputs = keras.Input(shape=IMG_SHAPE)

    # EfficientNetB0 applies its own preprocessing internally (no Rescaling needed)
    backbone = EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs,
        pooling="avg"           # GlobalAveragePooling already applied
    )

    if not trainable_backbone:
        backbone.trainable = False
    else:
        backbone.trainable = True
        if unfreeze_from is not None:
            # Freeze all layers except the last |unfreeze_from|
            for layer in backbone.layers[:unfreeze_from]:
                layer.trainable = False

    # Custom classification head
    x = backbone.output
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.40)(x)
    x = layers.Dense(64, activation="relu",
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output")(x)

    model = Model(inputs, outputs, name="EfficientNetB0_CrackDetector")
    return model, backbone

# Instantiate Phase 1 model (frozen backbone)
eff_model, eff_backbone = build_efficientnet(trainable_backbone=False)

total     = len(eff_model.trainable_variables)
non_train = len(eff_model.non_trainable_variables)
print(f"Trainable params   : {sum(np.prod(v.shape) for v in eff_model.trainable_variables):,}")
print(f"Non-trainable params: {sum(np.prod(v.shape) for v in eff_model.non_trainable_variables):,}")
eff_model.summary(line_length=90)

In [ ]:
# ── Phase 1 training — head only ──────────────────────────────────────────────

eff_model.compile(
    optimizer=keras.optimizers.Adam(P1_LR),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ]
)

ckpt_p1 = os.path.join(SAVE_DIR, "eff_phase1_best.keras")

cbs_p1 = [
    ModelCheckpoint(ckpt_p1, monitor="val_recall", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_recall", patience=8,
                  mode="max", restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4,
                      min_lr=1e-6, verbose=1),
]

history_p1 = eff_model.fit(
    train_ds,
    epochs=P1_EPOCHS,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=cbs_p1,
    verbose=1
)

print("\nPhase 1 complete — best val recall:",
      max(history_p1.history["val_recall"]))

In [ ]:
# ── Phase 1 training curves ────────────────────────────────────────────────────

def plot_history(history, title="Training History", save_name="history.png"):
    metrics = ["loss", "accuracy", "recall", "auc"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    axes = axes.ravel()
    for i, metric in enumerate(metrics):
        train_m = history.history.get(metric, [])
        val_m   = history.history.get(f"val_{metric}", [])
        axes[i].plot(train_m, label=f"Train {metric}", linewidth=2)
        axes[i].plot(val_m,   label=f"Val {metric}",   linewidth=2, linestyle="--")
        axes[i].set_title(metric.capitalize(), fontsize=12)
        axes[i].set_xlabel("Epoch")
        axes[i].legend()
        axes[i].grid(alpha=0.3)
    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, save_name), dpi=150)
    plt.show()

plot_history(history_p1, title="Phase 1 — Feature Extraction (Frozen Backbone)",
             save_name="phase1_history.png")

In [ ]:
# ── Phase 2 — Fine-tune top layers of backbone ────────────────────────────────
print(f"Total backbone layers : {len(eff_backbone.layers)}")
print(f"Unfreezing last {abs(UNFREEZE_FROM)} layers...")

# Rebuild model with partial unfreeze
eff_model_ft, _ = build_efficientnet(
    trainable_backbone=True,
    unfreeze_from=UNFREEZE_FROM
)

# Load best Phase 1 weights into the new model
eff_model_ft.load_weights(ckpt_p1)

eff_model_ft.compile(
    optimizer=keras.optimizers.Adam(P2_LR),
    loss="binary_crossentropy",
    metrics=[
        keras.metrics.BinaryAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.AUC(name="auc"),
    ]
)

ckpt_p2 = os.path.join(SAVE_DIR, "eff_phase2_best.keras")

cbs_p2 = [
    ModelCheckpoint(ckpt_p2, monitor="val_recall", mode="max",
                    save_best_only=True, verbose=1),
    EarlyStopping(monitor="val_recall", patience=10,
                  mode="max", restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=5,
                      min_lr=1e-7, verbose=1),
]

history_p2 = eff_model_ft.fit(
    train_ds,
    epochs=P2_EPOCHS,
    validation_data=val_ds,
    class_weight=CLASS_WEIGHTS,
    callbacks=cbs_p2,
    verbose=1
)

print("\nPhase 2 complete — best val recall:",
      max(history_p2.history["val_recall"]))

In [ ]:
# ── Phase 2 training curves ────────────────────────────────────────────────────
plot_history(history_p2, title="Phase 2 — Fine-Tuning (Top Backbone Layers Unfrozen)",
             save_name="phase2_history.png")

# Combined plot (P1 + P2 concatenated)
combined = {}
for k in ["loss","val_loss","recall","val_recall","auc","val_auc"]:
    combined[k] = history_p1.history.get(k, []) + history_p2.history.get(k, [])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (train_k, val_k, title) in zip(axes, [
    ("loss",   "val_loss",   "Loss"),
    ("recall", "val_recall", "Recall (Damaged)"),
    ("auc",    "val_auc",    "AUC"),
]):
    axes_list = combined[train_k]
    line_len  = len(history_p1.history.get(train_k, []))
    ax.axvline(line_len - 1, color="purple", linestyle=":", alpha=0.6, label="P1→P2 boundary")
    ax.plot(combined[train_k], label=f"Train {title}", linewidth=2)
    ax.plot(combined[val_k],   label=f"Val {title}",   linewidth=2, linestyle="--")
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle("Combined: Phase 1 + Phase 2", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "combined_history.png"), dpi=150)
plt.show()

## Step 8 — Decision Threshold Optimisation

The default threshold (0.5) is not optimal when the cost of a False Negative (missed crack) is much higher than a False Positive.

We sweep thresholds from 0.05 → 0.95 on the **validation set** and choose the threshold that maximises **F1 score for the Damaged class**.

In [ ]:
# ── Collect validation predictions ────────────────────────────────────────────

def get_predictions(model, dataset):
    """Return (probabilities, true_labels) numpy arrays from a tf.data.Dataset."""
    all_probs, all_labels = [], []
    for images, labels in dataset:
        probs = model(images, training=False).numpy().ravel()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy().ravel())
    return np.array(all_probs), np.array(all_labels)

val_probs, val_labels = get_predictions(eff_model_ft, val_ds)

print(f"Val samples : {len(val_labels)}")
print(f"Val prob range: [{val_probs.min():.4f}, {val_probs.max():.4f}]")

# ── Sweep thresholds ──────────────────────────────────────────────────────────
thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores, precisions, recalls = [], [], []

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f1  = f1_score(val_labels, preds, pos_label=1, zero_division=0)
    pre = precision_recall_curve(val_labels, val_probs)
    f1_scores.append(f1)
    from sklearn.metrics import precision_score, recall_score
    precisions.append(precision_score(val_labels, preds, pos_label=1, zero_division=0))
    recalls.append(recall_score(val_labels, preds, pos_label=1, zero_division=0))

best_idx   = int(np.argmax(f1_scores))
THRESHOLD  = float(thresholds[best_idx])
best_f1    = f1_scores[best_idx]

print(f"\nOptimal threshold : {THRESHOLD:.2f}")
print(f"  → Precision (Damaged) : {precisions[best_idx]:.4f}")
print(f"  → Recall    (Damaged) : {recalls[best_idx]:.4f}")
print(f"  → F1        (Damaged) : {best_f1:.4f}")

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thresholds, f1_scores,   label="F1 (Damaged)",        linewidth=2)
ax.plot(thresholds, precisions,  label="Precision (Damaged)",  linewidth=2, linestyle="--")
ax.plot(thresholds, recalls,     label="Recall (Damaged)",     linewidth=2, linestyle=":")
ax.axvline(THRESHOLD, color="red", linestyle="-.", linewidth=1.5,
           label=f"Optimal threshold = {THRESHOLD:.2f}")
ax.set_xlabel("Threshold")
ax.set_title("Threshold vs. Precision / Recall / F1 (on Validation Set)", fontsize=12)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "threshold_optimisation.png"), dpi=150)
plt.show()

## Step 9 — Full Test-Set Evaluation

We now evaluate **both models** (Baseline CNN and EfficientNetB0 fine-tuned) on the held-out test set using the optimal threshold found above.

In [ ]:
# ── Test-set evaluation helper ────────────────────────────────────────────────

def evaluate_model(model, test_dataset, threshold, model_name="Model"):
    probs, true_labels = get_predictions(model, test_dataset)
    preds = (probs >= threshold).astype(int)

    print(f"\n{'='*60}")
    print(f"  {model_name}  |  Threshold = {threshold:.2f}")
    print(f"{'='*60}")
    print(classification_report(true_labels, preds,
                                target_names=CLASS_NAMES, digits=4))

    # Confusion matrix
    cm_vals = confusion_matrix(true_labels, preds)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm_vals, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
    axes[0].set_title(f"Confusion Matrix — {model_name}")
    axes[0].set_ylabel("True Label")
    axes[0].set_xlabel("Predicted Label")

    # ROC curve
    fpr, tpr, _ = roc_curve(true_labels, probs)
    roc_auc     = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
    axes[1].plot([0,1], [0,1], "k--", linewidth=1)
    axes[1].set_xlabel("False Positive Rate")
    axes[1].set_ylabel("True Positive Rate")
    axes[1].set_title(f"ROC Curve — {model_name}")
    axes[1].legend(loc="lower right")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    safe_name = model_name.lower().replace(" ", "_")
    plt.savefig(os.path.join(SAVE_DIR, f"{safe_name}_evaluation.png"), dpi=150)
    plt.show()

    return probs, preds, true_labels

# Evaluate Baseline CNN (at default 0.5 — it has no threshold optimisation)
cnn_probs, cnn_preds, test_labels = evaluate_model(
    cnn_baseline, test_ds, threshold=0.5, model_name="Baseline CustomCNN"
)

In [ ]:
# ── Evaluate EfficientNetB0 fine-tuned (at optimised threshold) ───────────────
eff_probs, eff_preds, _ = evaluate_model(
    eff_model_ft, test_ds,
    threshold=THRESHOLD,
    model_name="EfficientNetB0 Fine-Tuned"
)

# ── Head-to-head comparison table ────────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def metric_row(name, true, probs, preds):
    return {
        "Model"    : name,
        "Accuracy" : accuracy_score(true, preds),
        "Precision": precision_score(true, preds, pos_label=1, zero_division=0),
        "Recall"   : recall_score(true, preds, pos_label=1, zero_division=0),
        "F1"       : f1_score(true, preds, pos_label=1, zero_division=0),
        "AUC-ROC"  : auc(*roc_curve(true, probs)[:2]),
    }

rows = [
    metric_row("Baseline CNN (t=0.50)", test_labels, cnn_probs, cnn_preds),
    metric_row(f"EfficientNetB0 FT (t={THRESHOLD:.2f})", test_labels, eff_probs, eff_preds),
]

df_compare = pd.DataFrame(rows).set_index("Model")
df_compare = df_compare.applymap(lambda x: f"{x:.4f}")
print("\nComparison on Test Set:")
print(df_compare.to_string())

## Step 10 — Grad-CAM Saliency Visualisation

Grad-CAM shows **which regions of the image the model focused on** when making its prediction.  
- If the model attends to the **specimen surface** → good signal
- If it attends to **fixtures or background** → ROI cropping needed in Phase 2

In [ ]:
# ── Grad-CAM implementation ───────────────────────────────────────────────────

def get_gradcam_heatmap(model, img_array, last_conv_layer_name):
    """
    Compute Grad-CAM heatmap for a single image (batch size = 1).
    img_array : shape (1, H, W, 3)
    """
    # Build a sub-model: input → last conv layer output + final model output
    grad_model = Model(
        inputs=model.inputs,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        # Sigmoid output: gradient of the single output neuron
        loss = predictions[:, 0]

    grads       = tape.gradient(loss, conv_outputs)        # (1, h, w, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))   # (C,)
    conv_outputs = conv_outputs[0]                          # (h, w, C)
    heatmap      = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)                      # (h, w)
    heatmap      = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_array, heatmap, alpha=0.4):
    """Overlay the Grad-CAM heatmap on the original image."""
    img = img_array[0].astype("uint8")
    heatmap_resized = np.uint8(255 * heatmap)
    jet  = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]
    jet_heatmap = tf.image.resize(jet_heatmap, [IMG_SIZE, IMG_SIZE]).numpy()
    jet_heatmap = np.uint8(255 * jet_heatmap)
    overlay = np.uint8(jet_heatmap * alpha + img * (1 - alpha))
    return overlay


# Find the last conv layer in EfficientNetB0
# (the last activation layer before GAP — typically "top_activation")
last_conv_layer = None
for layer in reversed(eff_model_ft.layers):
    if isinstance(layer, (layers.Conv2D, layers.DepthwiseConv2D)) or \
       "activation" in layer.name.lower():
        last_conv_layer = layer.name
        break

# Better: use the known EfficientNetB0 architecture landmark
# The last feature activation before GlobalAveragePooling is "top_activation"
try:
    eff_model_ft.get_layer("top_activation")
    last_conv_layer = "top_activation"
except ValueError:
    pass  # keep whatever was found above

print(f"Using Grad-CAM target layer: {last_conv_layer}")

In [ ]:
# ── Grad-CAM visualisation — 2 examples per class ────────────────────────────

n_examples = 2   # examples per class (Damaged + Undamaged)
fig, axes = plt.subplots(n_examples * 2, 3, figsize=(16, n_examples * 2 * 4))
fig.suptitle("Grad-CAM Saliency Maps  |  Model: EfficientNetB0 Fine-Tuned",
             fontsize=13, fontweight="bold")

col_titles = ["Original", "Grad-CAM Heatmap", "Overlay"]
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=11, fontweight="bold")

row = 0
for class_label, class_name in [(1, "Damaged"), (0, "Undamaged")]:
    sample_paths = [p for p, l in test_data if l == class_label][:n_examples]

    for path in sample_paths:
        raw  = tf.io.read_file(path)
        img  = tf.image.decode_jpeg(raw, channels=3)
        img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img  = tf.cast(img, tf.float32)
        img_batch = tf.expand_dims(img, axis=0)   # (1, 224, 224, 3)

        prob    = float(eff_model_ft(img_batch, training=False).numpy()[0, 0])
        pred_lbl = "Damaged" if prob >= THRESHOLD else "Undamaged"
        correct  = "✓" if pred_lbl == class_name else "✗"

        heatmap = get_gradcam_heatmap(eff_model_ft, img_batch, last_conv_layer)
        overlay = overlay_gradcam(img_batch.numpy(), heatmap, alpha=0.45)

        img_uint8 = img.numpy().clip(0, 255).astype("uint8")
        heatmap_resized = np.uint8(255 * tf.image.resize(
            heatmap[..., np.newaxis], [IMG_SIZE, IMG_SIZE]).numpy().squeeze())
        heatmap_rgb = np.uint8(255 * cm.jet(heatmap_resized / 255.0)[:, :, :3])

        for col_idx, img_to_show in enumerate([img_uint8, heatmap_rgb, overlay]):
            axes[row][col_idx].imshow(img_to_show)
            if col_idx == 0:
                axes[row][col_idx].set_ylabel(
                    f"True: {class_name}\nPred: {pred_lbl} {correct}\n(p={prob:.3f})",
                    fontsize=9, labelpad=5)
            axes[row][col_idx].axis("off")
        row += 1

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "gradcam_visualisation.png"), dpi=120)
plt.show()

## Step 11 — Prediction on a Single New Image

Use this cell to run inference on any new image file (e.g., a frame from a new UTM test).

In [ ]:
# ── Single-image inference ────────────────────────────────────────────────────

def predict_image(image_path, model=eff_model_ft, threshold=THRESHOLD,
                  show_gradcam=True):
    """
    Run inference on a single image file.
    Prints the prediction and optionally shows the Grad-CAM overlay.
    """
    raw  = tf.io.read_file(image_path)
    img  = tf.image.decode_jpeg(raw, channels=3)
    img  = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img  = tf.cast(img, tf.float32)
    img_batch = tf.expand_dims(img, axis=0)

    prob     = float(model(img_batch, training=False).numpy()[0, 0])
    pred_lbl = CLASS_NAMES[int(prob >= threshold)]

    print(f"Image    : {os.path.basename(image_path)}")
    print(f"Probability (Damaged) : {prob:.4f}")
    print(f"Threshold             : {threshold:.2f}")
    print(f"Prediction            : >>> {pred_lbl} <<<")

    if show_gradcam:
        heatmap = get_gradcam_heatmap(model, img_batch, last_conv_layer)
        overlay = overlay_gradcam(img_batch.numpy(), heatmap)
        img_uint8 = img.numpy().clip(0, 255).astype("uint8")
        heatmap_rgb = np.uint8(255 * cm.jet(
            np.uint8(255 * tf.image.resize(heatmap[..., np.newaxis],
                                            [IMG_SIZE, IMG_SIZE]).numpy().squeeze()) / 255.0
        )[:, :, :3])

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        for ax, im, title in zip(axes,
                                  [img_uint8, heatmap_rgb, overlay],
                                  ["Original", "Grad-CAM", "Overlay"]):
            ax.imshow(im)
            ax.set_title(title)
            ax.axis("off")
        fig.suptitle(f"Prediction: {pred_lbl}  (p={prob:.4f})", fontsize=13,
                     fontweight="bold", color="red" if pred_lbl=="Damaged" else "green")
        plt.tight_layout()
        plt.show()

    return pred_lbl, prob


# ── Example usage — change this path to any image ────────────────────────────
EXAMPLE_IMAGE = test_data[0][0]   # first test image
predict_image(EXAMPLE_IMAGE)

## Step 12 — Save Model to Google Drive

In [ ]:
# ── Save the fine-tuned EfficientNetB0 ───────────────────────────────────────

# SavedModel format (recommended for TF Serving / TFLite export)
saved_model_path = os.path.join(SAVE_DIR, "crack_detector_efficientnet")
eff_model_ft.export(saved_model_path)
print(f"SavedModel saved  : {saved_model_path}")

# Keras H5 format (for easy sharing and loading)
h5_path = os.path.join(SAVE_DIR, "crack_detector_efficientnet.h5")
eff_model_ft.save(h5_path)
print(f"H5 model saved    : {h5_path}")

# Also save the baseline CNN
cnn_h5_path = os.path.join(SAVE_DIR, "crack_detector_cnn_baseline.h5")
cnn_baseline.save(cnn_h5_path)
print(f"Baseline CNN saved: {cnn_h5_path}")

# Save the optimal threshold alongside the model
import json
meta = {
    "optimal_threshold" : float(THRESHOLD),
    "class_names"       : CLASS_NAMES,
    "img_size"          : IMG_SIZE,
    "model"             : "EfficientNetB0",
    "phase"             : "Phase1+Phase2 fine-tune",
}
meta_path = os.path.join(SAVE_DIR, "model_metadata.json")
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved    : {meta_path}")

print("\nAll files saved to Google Drive successfully.")

## Step 13 — Load Saved Model & Run Inference (Optional Reload Check)

Run this section independently (e.g., in a new Colab session) to verify the saved model loads correctly and produces the expected predictions.

In [ ]:
# ── Reload model + metadata and run a quick smoke test ───────────────────────
import json, tensorflow as tf

DRIVE_ROOT   = "/content/drive/MyDrive"
SAVE_DIR     = f"{DRIVE_ROOT}/saved_models"

# Load metadata
with open(f"{SAVE_DIR}/model_metadata.json") as f:
    meta = json.load(f)

THRESHOLD_LOADED = meta["optimal_threshold"]
CLASS_NAMES_LOADED = meta["class_names"]
IMG_SIZE_LOADED  = meta["img_size"]
print("Metadata loaded:", meta)

# Load model
loaded_model = tf.keras.models.load_model(f"{SAVE_DIR}/crack_detector_efficientnet.h5")
print("Model loaded successfully.")
print("Input shape :", loaded_model.input_shape)
print("Output shape:", loaded_model.output_shape)

# Quick smoke test on one image from Drive
import os, pathlib
test_img_path = str(sorted(
    pathlib.Path(f"{DRIVE_ROOT}/Dataset/Damaged").glob("*.jpg")
)[0])

raw  = tf.io.read_file(test_img_path)
img  = tf.image.decode_jpeg(raw, channels=3)
img  = tf.image.resize(img, [IMG_SIZE_LOADED, IMG_SIZE_LOADED])
img  = tf.cast(img, tf.float32)
img_batch = tf.expand_dims(img, 0)

prob = float(loaded_model(img_batch, training=False).numpy()[0, 0])
pred = CLASS_NAMES_LOADED[int(prob >= THRESHOLD_LOADED)]
print(f"\nSmoke test — {os.path.basename(test_img_path)}")
print(f"  P(Damaged) = {prob:.4f}  →  Predicted: {pred}")

## Step 14 — Final Results Summary

Run this cell **after** completing Steps 6–9. It consolidates every key metric into one printable block so you can share the output for diagnosis.

In [ ]:
# ── Step 14: Final Results Summary ───────────────────────────────────────────
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("=" * 65)
print("  FINAL RESULTS SUMMARY — Crack Detection")
print("=" * 65)

# ── Dataset split sizes ───────────────────────────────────────────────────────
print("\n[ Dataset Split ]")
for split, name in [(train_data,"Train"), (val_data,"Val"), (test_data,"Test")]:
    dmg = sum(1 for _, l in split if l == 1)
    und = sum(1 for _, l in split if l == 0)
    print(f"  {name:5s}: {len(split):4d} total  |  Damaged={dmg}  Undamaged={und}")

# ── Training epochs ───────────────────────────────────────────────────────────
print("\n[ Training Epochs Run ]")
print(f"  Baseline CNN      : {len(hist_cnn.history['loss'])} epochs")
print(f"  EfficientNet P1   : {len(history_p1.history['loss'])} epochs")
print(f"  EfficientNet P2   : {len(history_p2.history['loss'])} epochs")

# ── Optimal threshold ─────────────────────────────────────────────────────────
print(f"\n[ Optimal Decision Threshold (from val F1 sweep) ]")
print(f"  THRESHOLD = {THRESHOLD:.2f}")

# ── Per-model metrics on test set ─────────────────────────────────────────────
print("\n[ Test-Set Metrics ]")
header = f"  {'Model':<40} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}"
print(header)
print("  " + "-" * 68)

for model_name, probs, preds in [
    ("Baseline CNN (t=0.50)", cnn_probs, cnn_preds),
    (f"EfficientNetB0 FT (t={THRESHOLD:.2f})", eff_probs, eff_preds),
]:
    acc  = accuracy_score(test_labels, preds)
    prec = precision_score(test_labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(test_labels, preds, pos_label=1, zero_division=0)
    f1   = f1_score(test_labels, preds, pos_label=1, zero_division=0)
    roc  = roc_auc_score(test_labels, probs)
    print(f"  {model_name:<40} {acc:>6.4f} {prec:>6.4f} {rec:>6.4f} {f1:>6.4f} {roc:>6.4f}")

# ── Confusion matrix breakdown for EfficientNetB0 ────────────────────────────
print("\n[ EfficientNetB0 Confusion Matrix (Test Set) ]")
cm_vals = confusion_matrix(test_labels, eff_preds)
tn, fp, fn, tp = cm_vals.ravel()
print(f"  TP (crack correctly detected)      : {tp}")
print(f"  TN (no-crack correctly rejected)   : {tn}")
print(f"  FP (false alarm — no crack, pred crack) : {fp}")
print(f"  FN (CRITICAL — crack MISSED)       : {fn}")

# ── Verdict ───────────────────────────────────────────────────────────────────
print("\n[ Verdict ]")
eff_rec = recall_score(test_labels, eff_preds, pos_label=1, zero_division=0)
if fn == 0:
    verdict = "EXCELLENT — zero missed cracks on test set"
elif eff_rec >= 0.90:
    verdict = "GOOD — recall ≥ 90%, few cracks missed"
elif eff_rec >= 0.75:
    verdict = "ACCEPTABLE — but consider lowering threshold or adding ROI crop"
else:
    verdict = "NEEDS IMPROVEMENT — too many cracks missed"

print(f"  {verdict}")
print("\n" + "=" * 65)
print("  Share this output to decide next improvement steps.")
print("=" * 65)
